In [0]:
df = spark.read.csv("/Volumes/workspace/default/ev_data/ev_sessions_raw.csv", header=True, inferSchema=True)

#df.printSchema() #shcema check -- oversvability pillar 3
#df.count()      #volume check -- oversvability pillar 2
#display(df.limit(10))

#df.filter(df.city == "London")

#Select * from /Volumes/workspace/default/ev_data/ where city = "London"

df.createOrReplaceTempView("sessions")
spark.sql("select * from sessions where city == 'London'").display()


session_id,session_date,charger_id,city,kwh_delivered,revenue_gbp,charger_status
100004,2026-03-01,CK-0170,London,18.16,9.93,completed
100018,2026-03-01,CK-0218,London,24.81,11.25,completed
100019,2026-03-01,CK-0207,London,23.57,9.98,completed
100021,2026-03-01,CK-0231,London,15.91,8.24,completed
100024,2026-03-01,CK-0348,London,15.59,8.39,completed
100025,2026-03-01,CK-0283,London,23.26,12.24,completed
100029,2026-03-01,CK-0113,London,27.83,15.3,completed
100030,2026-03-01,CK-0343,London,17.55,9.44,completed
100032,2026-03-01,CK-0184,London,12.66,6.52,completed
100034,2026-03-01,CK-0187,London,14.9,6.34,completed


In [0]:
df = spark.read.csv("/Volumes/workspace/default/ev_data/ev_sessions_raw.csv", header=True, inferSchema=True)

import pyspark.sql.functions as F

df = (df.withColumn("revenue_valikd",F.col("revenue_gbp")>0)
        .withColumn("session_date", F.to_date("session_date"))     
        .withColumn("is_weekend",F.dayofweek("session_date").isin(1,7))
        .withColumn("kwh_bank",F.when(F.col("kwh_delivered") < 10, "small").when(F.col("kwh_delivered") > 30, "large").otherwise("medium")))

df.printSchema() 
display(df.limit(10))



root
 |-- session_id: integer (nullable = true)
 |-- session_date: date (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- kwh_delivered: double (nullable = true)
 |-- revenue_gbp: double (nullable = true)
 |-- charger_status: string (nullable = true)
 |-- revenue_valikd: boolean (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- kwh_bank: string (nullable = false)



session_id,session_date,charger_id,city,kwh_delivered,revenue_gbp,charger_status,revenue_valikd,is_weekend,kwh_bank
100001,2026-03-01,CK-0107,Bristol,17.03,8.47,completed,true,true,medium
100002,2026-03-01,CK-0373,Manchester,17.34,8.88,completed,true,true,medium
100003,2026-03-01,CK-0258,Bristol,11.5,6.23,completed,true,true,medium
100004,2026-03-01,CK-0170,London,18.16,9.93,completed,true,true,medium
100005,2026-03-01,CK-0446,Bristol,15.01,6.69,completed,true,true,medium
100006,2026-03-01,CK-0307,Manchester,14.19,6.79,completed,true,true,medium
100007,2026-03-01,CK-0089,Glasgow,18.78,9.86,completed,true,true,medium
100008,2026-03-01,CK-0367,Manchester,20.04,10.01,completed,true,true,medium
100009,2026-03-01,CK-0035,Glasgow,13.55,6.15,completed,true,true,medium
100010,2026-03-01,CK-0106,Manchester,19.46,8.25,completed,true,true,medium


In [0]:
df = spark.read.csv("/Volumes/workspace/default/ev_data/ev_sessions_raw.csv", header=True, inferSchema=True)

import pyspark.sql.functions as F

# df = df.groupBy("session_date").agg(F.count_distinct("session_id").alias("sessions"),
#                                     F.round(F.sum("revenue_gbp"),2).alias("revenue"),
#                                     F.round(F.sum("kwh_delivered"),2).alias("kwh")).orderBy("session_date") 



df = df.groupBy("session_date","city").agg(F.count_distinct("session_id").alias("sessions"),
                                        F.round(F.sum("revenue_gbp"),2).alias("revenue"),
                                        F.round(F.sum("kwh_delivered"),2).alias("kwh")).orderBy("session_date","city") 
display(df.limit(10))
    
            


session_date,city,sessions,revenue,kwh
2026-03-01,Bristol,156,1388.39,2872.01
2026-03-01,Glasgow,101,949.16,1959.64
2026-03-01,London,400,3550.03,7273.12
2026-03-01,Manchester,212,1811.82,3693.78
2026-03-02,Bristol,217,1884.65,3930.18
2026-03-02,Glasgow,115,1000.39,2043.95
2026-03-02,London,540,4578.37,9494.46
2026-03-02,Manchester,287,2490.24,5092.1
2026-03-03,Bristol,189,1541.43,3231.97
2026-03-03,Glasgow,114,953.01,1947.16
